# MURI: Modelling actuator self-assembly

## Imports statements and setup

First, import statements

In [ ]:
import numpy as np
import pyvista as pv
import polyscope as ps
import fabsim_py
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter
import IPython.display
import csv
import cv2
import igl

In [ ]:
# Folder stuff, change for different data sets

data_path = "data/"
# data_path = "data/3.5hrs/"
# data_path = "data/3 post/"
# data_path = "data/7D/"

Constants

In [ ]:
dt = 1 / 24
k0 = 1e-4
k1 = 4e-4
kd = 1e-3 # rate of dissociation
e0 = 1.2e-1
e1 = 1.7e-1
frac_f = 0.7
frac_s = 0.25
n = 16
world_coords_to_px = 1287.2
k_tension = 0

Fix DOFs

In [ ]:
fixed_dofs = []

def fix_dofs_in_circle(radius, center, V):
  for i in range(V.shape[0]):
    if np.linalg.norm(center - V[i, :2]) < radius + 1e-6:
      # if np.linalg.norm(V[i, :2]) > np.linalg.norm(center):
      # fixed_dofs.append(i)
      if center[0] > 0:
        if V[i, 0] - center[0] > 0:
          fixed_dofs.append(i)
        elif V[i, 1]**2 > -(V[i, 0] - center[0]) * V[i, 0]:
          fixed_dofs.append(i)
      else:
        if V[i, 0] - center[0] < 0:
          fixed_dofs.append(i)
        elif V[i, 1]**2 > -(V[i, 0] - center[0]) * V[i, 0]:
          fixed_dofs.append(i)

Display compaction with polar plots

In [ ]:
def polygons_polar_plot(V, polymer_frac, scale):
  n = polymer_frac.shape[1]
  polygon = []
  rotated_polygon = []
  for i in range(n):
    angle = np.pi / n * i
    polygon.append(np.array([np.cos(angle), np.sin(angle)]))
    rotated_polygon.append(np.array([np.cos(angle + np.pi), np.sin(angle + np.pi)]))
  polygon = scale * np.array(polygon)
  rotated_polygon = scale * np.array(rotated_polygon)

  # faces = np.arange(F.shape[0] * 2 * n).reshape(F.shape[0], 2 * n)
  polygon_face = np.zeros((2 * n, 3))
  polygon_face[:, 1] = np.arange(1, 2 * n + 1)
  polygon_face[:, 2] = np.arange(2, 2 * n + 2)
  polygon_face[2 * n - 1, 2] = 1

  verts = []
  faces = []
  for i in range(F.shape[0]):
    center = (V[F[i, 0], :] + V[F[i, 1], :] + V[F[i, 2], :]) / 3
    verts.append(center.reshape((1, 2)))
    verts.append(polygon * polymer_frac[i, :][:, None] + center)
    verts.append(rotated_polygon * polymer_frac[i, :][:, None] + center)
    faces.append(polygon_face + i * (2 * n + 1))
  verts = np.concatenate(verts, axis=0)
  verts = np.hstack((verts, 0.1 * np.ones((verts.shape[0], 1))))
  faces = np.concatenate(faces, axis=0)

  return verts, faces

def vertex_polar_plot(V, polymer_frac, scale):
  n = polymer_frac.shape[1]
  polygon = []
  rotated_polygon = []
  for i in range(n):
    angle = np.pi / n * i
    polygon.append(np.array([np.cos(angle), np.sin(angle)]))
    rotated_polygon.append(np.array([np.cos(angle + np.pi), np.sin(angle + np.pi)]))
  polygon = scale * np.array(polygon)
  rotated_polygon = scale * np.array(rotated_polygon)

  # faces = np.arange(F.shape[0] * 2 * n).reshape(F.shape[0], 2 * n)
  polygon_face = np.zeros((2 * n, 3))
  polygon_face[:, 1] = np.arange(1, 2 * n + 1)
  polygon_face[:, 2] = np.arange(2, 2 * n + 2)
  polygon_face[2 * n - 1, 2] = 1

  verts = []
  faces = []
  for i in range(V.shape[0]):
    verts.append(V[i, :].reshape((1, 2)))
    verts.append(polygon * polymer_frac[i, :][:, None] + V[i, :])
    verts.append(rotated_polygon * polymer_frac[i, :][:, None] + V[i, :])
    faces.append(polygon_face + i * (2 * n + 1))
  verts = np.concatenate(verts, axis=0)
  verts = np.hstack((verts, 0.1 * np.ones((verts.shape[0], 1))))
  faces = np.concatenate(faces, axis=0)

  return verts, faces

CSV data processing

In [ ]:
def read_csv_to_array(filename):
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        data = [list(map(float, row)) for row in reader]
    return np.array(data, dtype=np.float32)

def to_8bit_rgb(x):
    R = np.floor(x)
    G = np.floor((x - R) * 256)
    B = np.floor(((x - R) * 256 - G) * 256)
    return R, G, B

def from_8bit_rgb(R, G, B):
    return R + G / 256. + B / 256. / 256.

In [ ]:
def add_mirror_images(target, source, source_flipped, height, width):
  n_rows = target.shape[0]

  target[n_rows - height//2:n_rows, 0:width//2] += np.flip(source_flipped[0:height//2, 0:width//2], axis=1)
  target[n_rows - height//2:n_rows, 0:width//2] += np.flip(source[height//2:height, 0:width//2], axis=(0, 1))
  target[n_rows - height//2:n_rows, 0:width//2] += source[0:height//2, width//2:width]
  target[n_rows - height//2:n_rows, 0:width//2] += np.flip(source_flipped[height//2:height, width//2:width], axis=0)

## Polyscope init

In [ ]:
ps.init()
ps.load_color_map("twilight", "data/twilight_colormap.png");
ps.set_give_focus_on_show(True)
ps.set_ground_plane_mode("shadow_only")

ps.set_navigation_style("planar")
ps.set_view_projection_mode("orthographic")
ps.set_SSAA_factor(3)

Load initial mesh and display

In [ ]:
# mesh = pv.read("data/top_surface_3posts_highres.obj")
mesh = pv.read("data/top_surface_ultrahighres.obj")
P = np.array(mesh.points, dtype=np.float64)
F = np.array(mesh.faces, dtype=np.int32)
F = F.reshape((F.shape[0] // 4, 4))[:, 1:]
P = P[:,:2]

fixed_dofs = []

X = np.reshape(P, 2 * P.shape[0])
fix_dofs_in_circle(0.75, np.array([-2.275, 0]), P)
# X[2 * np.array(fixed_dofs)] += 2 * (2.275 - 2.2)
fix_dofs_in_circle(0.75, np.array([2.275, 0]), P)
# X[2 * np.array(fixed_dofs)] -= (2.275 - 2.2)
# P[:, 0] = np.reshape(X, P.shape)[:, 0]

# fix_dofs_in_circle(0.75, np.array([-2.275, 2.627/2]), P)
# fix_dofs_in_circle(0.75, np.array([2.275, 2.627/2]), P)
# fix_dofs_in_circle(0.75, np.array([0, -2.627]), P)

fixed_dofs = np.sort(fixed_dofs)

ps.set_view_from_json('{"farClipRatio":20.0,"fov":16.0,"nearClipRatio":0.005,"projectionMode":"Orthographic","viewMat":[1.0,0.0,0.0,0.0,0.0,0.997785151004791,0.0665190145373344,0.000246047973632812,0.0,0.0665190145373344,0.997785151004791,-25.5602951049805,0.0,0.0,0.0,1.0],"windowHeight":1964,"windowWidth":3456}')
# ps.set_view_from_json('{"fov":20.0,"nearClipRatio":0.005,"projectionMode":"Orthographic","viewMat":[1.0,0.0,0.0,0.0,0.0,0.997785151004791,-0.066519021987915,-1.22771632671356,0.0,0.066519021987915,0.997785151004791,-20.0573978424072,0.0,0.0,0.0,1.0],"windowHeight":783,"windowWidth":1440}')

ps_mesh = ps.register_surface_mesh("Initial mesh", P, F, smooth_shade=True, enabled=True, color=(42/255, 53/255, 213/255), edge_width=1, material="flat")
ps.set_view_from_json('{"fov":20.0')
ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 

## Load fiber orientation and intensity data

First simulate tissue without fibers for initial alignment guess

In [ ]:
P = P[:,:2]
V = P.copy()

k_post = 7

for i in range(9):
  sigma = 0
  stretch = 1.25 + 0.25 * i
  fabsim_py.simulate_membrane(V, P, F, np.array([]), np.array([]), np.array([[]]), fixed_dofs, stretch, 0.49, sigma, k_post, k_tension)

ps.remove_all_structures()
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

Load orientation data $(\eta, \theta)$ and transfer it to mesh

In [ ]:
img = cv2.imread(data_path + 'orientation_dispersion_combined.png', cv2.IMREAD_UNCHANGED)
dispersion_data = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * 0.5 / 256
dispersion_data *= img[:,:,3] / 255

img = cv2.imread(data_path + 'orientation_mean.png', cv2.IMREAD_UNCHANGED)
orientation_data = (256 - from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])) * np.pi / 256 # angles from the 'orientation.png' image have to be negated
orientation_data *= img[:,:,3] / 255

eta = fabsim_py.image_data_to_mesh(V, F, dispersion_data, world_coords_to_px)
orientation = fabsim_py.orientation_data_to_mesh(V, F, orientation_data, world_coords_to_px)
orientation /= np.max(np.linalg.norm(orientation, axis=1))

ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps_mesh.add_vector_quantity("mean vector orientation", orientation, defined_on='faces', enabled=False)
ps_mesh.add_scalar_quantity("eta measured", eta, defined_on='faces', vminmax = (0, 0.5), enabled=True)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

Simulate with fibers and realign data

In [ ]:
phi = 0.025 * np.clip(np.linalg.norm(orientation, axis=1), a_min=1e-6, a_max=1)
theta = np.atan2(orientation[:,1], orientation[:,0])

stretch = 2.9
sigma = 37
k_post = 11
fabsim_py.simulate_membrane(V, P, F, theta, eta, phi[:,None], fixed_dofs, stretch, 0.49, sigma, k_post, k_tension)

eta = fabsim_py.image_data_to_mesh(V, F, dispersion_data, world_coords_to_px)
orientation = fabsim_py.orientation_data_to_mesh(V, F, orientation_data, world_coords_to_px)
orientation /= np.max(np.linalg.norm(orientation, axis=1))
theta = np.atan2(orientation[:,1], orientation[:,0])
phi = 0.025 * np.ones_like(phi)

# fabsim_py.update_Phi(V, P, F, theta, eta, phi, fixed_dofs, stretch, 0.49, sigma)

fabsim_py.simulate_membrane(V, P, F, theta, eta, phi[:,None], fixed_dofs, stretch, 0.49, sigma, k_post, k_tension)
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps_mesh.add_vector_quantity("mean vector orientation", orientation, defined_on='faces', enabled=False)
ps_mesh.add_scalar_quantity("eta", eta, defined_on='faces', vminmax = (0, 0.5), enabled=True)

# dirs = fabsim_py.theta0(V, P, F, stretch)
# dirs[:,1] *= -1
# ps_mesh.add_vector_quantity("thetas", dirs, defined_on='faces', enabled=True)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
# img = cv2.imread(data_path + 'orientation-dispersion-combined.png', cv2.IMREAD_UNCHANGED)
# dispersion_data = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * 0.5 / 256
# dispersion_data *= img[:,:,3] / 255

# eta = fabsim_py.image_data_to_mesh(V, F, dispersion_data, world_coords_to_px)
# ps_mesh.add_scalar_quantity("new eta", eta, defined_on='faces', vminmax = (0, 0.5), enabled=True)
# ps.show()

Estimate volume fraction $\phi^p$ from orientation data $(\eta, \theta)$

In [ ]:
I5 = fabsim_py.I5(V, P, F, orientation, eta, stretch)

a = k1 / frac_f * sigma * (I5 - np.sqrt(I5))
b = k0 / frac_f + kd - 0.05 * a
c = -k0 * 0.05 / frac_f * np.ones_like(b)

phi_p = (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)

k_post = 12
fabsim_py.simulate_membrane(V, P, F, theta, eta, phi_p[:,None], fixed_dofs, stretch, 0.49, sigma, k_post, k_tension)

ps_mesh.update_vertex_positions(V)
ps_mesh.add_scalar_quantity("phi_p", phi_p, defined_on='faces', vminmax = (0, 0.05), enabled=True)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
read = cv2.imread('data/intensity.png')
intensity = from_8bit_rgb(read[:,:,0], read[:,:,1], read[:,:,2])

I = fabsim_py.image_data_to_mesh(V, F, intensity, world_coords_to_px)
I = I / np.max(I)

# phi = 0.025 * I

ps.remove_all_structures()
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps_mesh.add_scalar_quantity("intensity", I, defined_on='faces', enabled=True)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

## Optimize simulation parameters

Read SDF data

In [ ]:
read = cv2.imread('data/distance-boundary.png')
dist = from_8bit_rgb(read[:,:,0], read[:,:,1], read[:,:,2])

read = cv2.imread('data/distance-post.png')
dist2 = from_8bit_rgb(read[:,:,0], read[:,:,1], read[:,:,2])

In [ ]:
indices = igl.boundary_loop_all(F)

def distance(V, indices, dist, dist2):
  boundary_indices = indices[0]
  post_indices = indices[1] + indices[2]
  return (1 * fabsim_py.distance(V, boundary_indices, dist) + 1.0 * fabsim_py.distance(V, post_indices, dist2)) / (len(boundary_indices) + len(post_indices))

# def distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2):
#   grad = 1.0 * fabsim_py.distance_gradient(V, boundary_indices, dist_grad[0], dist_grad[1])
#   grad += 1.0 * fabsim_py.distance_gradient(V, indices[1], dist_grad2[0], dist_grad2[1])
#   grad += 1.0 * fabsim_py.distance_gradient(V, indices[2], dist_grad2[0], dist_grad2[1])
#   return grad

In [ ]:
stretch = 1 + (np.sqrt(22.4696/11.871) * 72**(0.274/2) - 1) / 0.71

def fun(x):
  I5 = fabsim_py.I5(V, P, F, orientation, eta, x[0])
  a = k1 / frac_f * stretch * (I5 - np.sqrt(I5))
  b = k0 / frac_f + kd - 0.05 * a
  c = -k0 * 0.05 / frac_f * np.ones_like(b)

  phi_p = (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)

  fabsim_py.simulate_membrane(V, P, F, theta, eta, phi_p[:,None], fixed_dofs, stretch, 0.49, x[0], k_post, k_tension)
  return distance(V, indices, dist, dist2)

print("Initial distance:", distance(V, indices, dist, dist2))

ret = minimize(fun, x0=np.array([42]), method='Nelder-Mead')
print(ret)

eta = fabsim_py.image_data_to_mesh(V, F, dispersion_data, world_coords_to_px)
orientation = fabsim_py.orientation_data_to_mesh(V, F, orientation_data, world_coords_to_px)
theta = np.atan2(orientation[:,1], orientation[:,0])

# stretch = ret.x[0]
sigma = ret.x[0]
k_post = 12
fabsim_py.simulate_membrane(V, P, F, theta, eta, phi_p[:,None], fixed_dofs, stretch, 0.49, sigma, k_post, k_tension)
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
print("Final distance:", distance(V, indices, dist, dist2))

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

## Fiber polymerization simulation

Compaction video

In [ ]:
sigma = 42
k_post = 7
V = P.copy()
Phi = np.zeros((2 * F.shape[0], 2))
ps.remove_all_structures()

P[:, 1] *= -1
ps_mesh = ps.register_surface_mesh("Simulation mesh", P, F, smooth_shade=True, enabled=True, color=(42/255, 53/255, 213/255), edge_width=1)
P[:, 1] *= -1


for i in range(1000):
  t_0 = (3 / 5)**(-1/0.649)
  stretch = 1 / np.sqrt(5 / 3 * (3.5 + (i+1) / 1000 * 72)**(-0.649))

  fabsim_py.implicit_euler(Phi, V, P, F, fixed_dofs, stretch, 0.49, sigma, k0, k1, kd, 0.1, 0.1, k_post, int(1000 / (i+1)), k_tension)
  
  V[:, 1] *= -1
  ps_mesh.update_vertex_positions(V)
  V[:, 1] *= -1

  phi_p = np.empty(F.shape[0])
  eta = np.empty(F.shape[0])

  for i in range(F.shape[0]):
    phi_p[i] = Phi[2 * i, 0] + Phi[2 * i + 1, 1]
    M = Phi[2 * i : 2 * (i + 1), :] / phi_p[i]
    w, v = np.linalg.eig(M)
    if w[0] > 0.5:
      eta[i] = w[1]
    else:
      eta[i] = w[0]

  ps_mesh.add_scalar_quantity("eta", eta, defined_on='faces', vminmax = (0, 0.5), enabled=True)
  # ps_mesh.add_scalar_quantity("phi_p", phi_p, defined_on='faces', vminmax = (0, 0.05), enabled=True)
  ps.screenshot()

Comparison data at 72h

In [ ]:
eta_measured = fabsim_py.image_data_to_mesh(V, F, dispersion_data, world_coords_to_px)
orientation = fabsim_py.orientation_data_to_mesh(V, F, orientation_data, world_coords_to_px)
orientation /= np.max(np.linalg.norm(orientation, axis=1))

I5 = fabsim_py.I5(V, P, F, orientation, eta_measured, stretch)

a = k1 / frac_f * sigma * (I5 - np.sqrt(I5))
b = k0 / frac_f + kd - 0.05 * a
c = -k0 * 0.05 / frac_f * np.ones_like(b)

phi_p_measured = (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)

phi_p = np.empty(F.shape[0])
eta = np.empty(F.shape[0])

for i in range(F.shape[0]):
  phi_p[i] = Phi[2 * i, 0] + Phi[2 * i + 1, 1]
  M = Phi[2 * i : 2 * (i + 1), :] / phi_p[i]
  w, v = np.linalg.eig(M)
  if w[0] > 0.5:
    eta[i] = w[1]
  else:
    eta[i] = w[0]

ps_mesh.add_scalar_quantity("eta", eta, defined_on='faces', vminmax = (0, 0.5), enabled=True)
ps_mesh.add_scalar_quantity("phi_p", phi_p, defined_on='faces', vminmax = (0, 0.05), enabled=True)
ps_mesh.add_scalar_quantity("phi_p_measured", phi_p_measured, defined_on='faces', enabled=True)
ps_mesh.add_scalar_quantity("eta_measured", eta_measured, defined_on='faces', enabled=True)
ps.show()

print(f"Distance with tissue: {distance(V, indices, dist, dist2) * 1000:.2f} microns")
print("eta difference:", np.average(np.abs(eta - eta_measured) / 0.5) * 100, "%")
print("phi difference:", np.average(np.abs(phi_p - phi_p_measured) / 0.05) * 100, "%")

Compaction at a set frame (3.5h)

In [ ]:
# P = P[:,:2]
# V = P.copy()
# Phi = np.zeros((2 * F.shape[0], 2))
# sigma = 42
# k1 = 4e-3
# kd = 1e-2
# k_post = 20
# k_tension = 0

# for i in range(9):
#   fabsim_py.simulate_membrane(V, P, F, np.array([]), np.array([]), np.array([[]]), fixed_dofs, 1.25 + 0.25 * i, 0.49, sigma, k_post, k_tension)

# # t_0 = (3 / 5)**(-1/0.649)
# # stretch = 1 / np.sqrt(5 / 3 * (3.5 + t_0)**(-0.649))

# stretch = np.sqrt(28.56 / 5.076666667)

# fabsim_py.simulate_membrane(V, P, F, np.array([]), np.array([]), np.array([[]]), fixed_dofs, stretch, 0.49, sigma, k_post, k_tension)
# fabsim_py.implicit_euler(Phi, V, P, F, fixed_dofs, stretch, 0.49, sigma, k0, k1, kd, 0.1, 0.1, k_post, 1000, k_tension)

# ps.remove_all_structures()
# V[:,1] *= -1
# ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1, material="flat")

# phi_p = np.empty(F.shape[0])
# eta = np.empty(F.shape[0])
# dirs = np.empty((F.shape[0], 2))

# for i in range(F.shape[0]):
#   phi_p[i] = Phi[2 * i, 0] + Phi[2 * i + 1, 1]
#   M = Phi[2 * i : 2 * (i + 1), :] / phi_p[i]
#   w, v = np.linalg.eig(M)
#   if w[0] > 0.5:
#     dirs[i,:] = v[0]
#     eta[i] = w[1]
#   else:
#     dirs[i,:] = v[1]
#     eta[i] = w[0]

# # dirs[:,1] *= -1
# angles = np.atan2(dirs[:,1], dirs[:,0])
# angles = np.mod(angles, np.pi)
# # angles = np.array([theta if theta < np.pi / 2 else theta - np.pi for theta in angles])
# J = fabsim_py.J(V,P,F,stretch)

# ps_mesh.add_scalar_quantity("phi_p", phi_p, defined_on='faces', vminmax = (0, 0.05), enabled=True)
# ps_mesh.add_scalar_quantity("eta", eta, defined_on='faces', vminmax = (0, 0.5), enabled=True, cmap="turbo")
ps_mesh.add_scalar_quantity("dirs", angles, defined_on='faces', enabled=False)
# ps_mesh.add_vector_quantity("dirs", dirs, defined_on='faces', enabled=False)

R, G, B = to_8bit_rgb(angles * 256 / np.pi)
colors_face = np.array([B.T, G.T, R.T]).T
ps_mesh.add_color_quantity("eta colors", colors_face / 256, defined_on='faces')

ps_mesh.add_scalar_quantity("J", J, defined_on='faces', enabled=False)

ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
img = np.array(cv2.imread('data/sim/72h_eta.png', cv2.IMREAD_UNCHANGED), dtype=np.float32)
converted = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])
converted = np.clip(converted, a_min=0, a_max=217)

eta = converted * 0.5 / 217

print(np.min(converted), np.max(converted))
print(np.min(eta), np.max(eta))

plt.imshow(eta, cmap='turbo', origin='lower', vmin=0, vmax=0.5, extent=[0, eta.shape[1] / world_coords_to_px, 0, eta.shape[0] / world_coords_to_px], aspect='equal', alpha=img[:,:,3] / 255)
plt.colorbar(label="$\eta$")
plt.title("Simulated compaction (72h)")
plt.tight_layout()
# # plt.show()

img = np.array(cv2.imread('data/sim/72h_theta.png', cv2.IMREAD_UNCHANGED), dtype=np.float32)
converted = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])
converted = np.clip(converted, a_min=0, a_max=217)

theta = converted * np.pi / 217

height, width = theta.shape

X, Y = np.meshgrid(np.arange(0, width, 99), np.arange(0, height, 99))
U = np.cos(theta) * img[:,:,3] / 255
V = -np.sin(theta) * img[:,:,3] / 255

q = plt.quiver(X / world_coords_to_px,  Y / world_coords_to_px, U[Y, X], V[Y, X], color="white", headlength=0, headaxislength=0, headwidth=0, scale=50, units='width')

plt.show()

In [ ]:
img = np.array(cv2.imread('data/orientation1_dispersion.png', cv2.IMREAD_UNCHANGED), dtype=np.float32)
converted = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])
converted = np.clip(converted, a_min=0, a_max=217)

eta = converted * 0.5 / 217

print(np.min(converted), np.max(converted))
print(np.min(eta), np.max(eta))

plt.imshow(eta, cmap='turbo', origin='lower', vmin=0, vmax=0.5, extent=[0, eta.shape[1] / world_coords_to_px, 0, eta.shape[0] / world_coords_to_px], aspect='equal', alpha=img[:,:,3] / 255)
plt.colorbar(label="$\eta$")
plt.title("Measured compaction (72h, S1)")
plt.tight_layout()

img = np.array(cv2.imread('data/orientation1_mean.png', cv2.IMREAD_UNCHANGED), dtype=np.float32)
converted = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])
converted = np.clip(converted, a_min=0, a_max=217)

theta = converted * np.pi / 256 + np.pi / 2

height, width = theta.shape

X, Y = np.meshgrid(np.arange(0, width, 99), np.arange(0, height, 99))
U = np.cos(theta) * img[:,:,3] / 255
V = np.sin(theta) * img[:,:,3] / 255

q = plt.quiver(X / world_coords_to_px,  Y / world_coords_to_px, U[Y, X], V[Y, X], color="grey", headlength=0, headaxislength=0, headwidth=0, scale=50, units='width')

plt.show()

## Fitting the ODE coefficients for fiber simulation

In [ ]:
eta_measured = fabsim_py.image_data_to_mesh(V, F, dispersion_data, world_coords_to_px)
orientation = fabsim_py.orientation_data_to_mesh(V, F, orientation_data, world_coords_to_px)
orientation /= np.max(np.linalg.norm(orientation, axis=1))
# theta = np.atan2(orientation[:,1], orientation[:,0])

I5 = fabsim_py.I5(V, P, F, orientation, eta_measured, stretch)

a = k1 / frac_f * sigma * (I5 - np.sqrt(I5))
b = k0 / frac_f + kd - 0.05 * a
c = -k0 * 0.05 / frac_f * np.ones_like(b)

phi_p_estimated = (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)

Phi_measured = np.empty((2 * F.shape[0], 2))

for i in range(F.shape[0]):
  Phi_measured[2*i : 2*(i+1), :] = phi_p_estimated[i] * (eta_measured[i] * np.array([[1,0], [0,1]]) + (1 - 2 * eta_measured[i]) * orientation[i,:].T.dot(orientation[i,:]))

In [ ]:
def fitting2(Phi_measured):
  def fun(params):
    Phi = np.zeros((2 * F.shape[0], 2))
    fabsim_py.phi_ode(Phi, V, P, F, stretch, 0.49, sigma, k0, params[0], params[1], 0.1, 10000)

    return (Phi - Phi_measured).flatten()

  initial_guess = np.array([5e-2, 0.1])
  return least_squares(fun, initial_guess, bounds=[[0, 0], [np.inf, np.inf]], verbose=2)

In [ ]:
def fitting(eta_measured):
  # I5 = fabsim_py.I5(V, P, F, orientation, eta_measured, stretch)

  # # re-estimate phi_p
  # a = k1 / frac_f * sigma * (I5 - np.sqrt(I5))
  # b = k0 / frac_f + kd - 0.05 * a
  # c = -k0 * 0.05 / frac_f * np.ones_like(b)

  # phi_p_estimated = (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)

  def fun(params):
    Phi = np.zeros((2 * F.shape[0], 2))
    # fabsim_py.phi_ode(Phi, V, P, F, stretch, 0.49, sigma, k0, params[0], params[1], 0.1, 10000)
    fabsim_py.implicit_euler(Phi, V, P, F, fixed_dofs, stretch, 0.49, sigma, k0, params[0], params[1], 0.1, 0.1, k_post, 500, k_tension)

    phi_p = np.empty(F.shape[0])
    eta = np.empty(F.shape[0])
    # dirs = np.empty((F.shape[0], 2))

    for i in range(F.shape[0]):
      phi_p[i] = Phi[2 * i, 0] + Phi[2 * i + 1, 1]
      if(phi_p[i] > 1e-8):
        M = Phi[2 * i : 2 * (i + 1), :] / phi_p[i]
      else:
        print(phi_p[i])
        print(Phi[2 * i : 2 * (i + 1), :])
        print(params)
        M = Phi[2 * i : 2 * (i + 1), :]
      w, v = np.linalg.eig(M)
      if w[0] > 0.5:
        # dirs[i,:] = v[0]
        eta[i] = w[1]
      else:
        # dirs[i,:] = v[1]
        eta[i] = w[0]

    # return np.concatenate((eta - eta_measured, 0 * (phi_p - phi_p_estimated)))
    return np.linalg.norm(eta - eta_measured)

  initial_guess = np.array([1e-3, 1e-3])
  # return least_squares(fun, initial_guess, bounds=[[0, 0], [np.inf, np.inf]], verbose=2)
  return minimize(fun, x0=initial_guess, method='Nelder-Mead', bounds=[(0, np.inf), (1e-3, np.inf)], options={"disp":True})

In [ ]:
P = P[:,:2]
V = P.copy()
Phi = np.zeros((2 * F.shape[0], 2))

for i in range(9):
  fabsim_py.simulate_membrane(V, P, F, np.array([]), np.array([]), np.array([[]]), fixed_dofs, 1.25 + 0.25 * i, 0.49, sigma, k_post, k_tension)

stretch = 1 + (np.sqrt(22.4696/11.871) * 72**(0.274/2) - 1) / 0.71

fabsim_py.simulate_membrane(V, P, F, np.array([]), np.array([]), np.array([[]]), fixed_dofs, stretch, 0.49, sigma, k_post, k_tension)
fabsim_py.implicit_euler(Phi, V, P, F, fixed_dofs, stretch, 0.49, sigma, k0, 1e-3, 1e-3, 0.1, 0.1, k_post, 1000, k_tension)

params = fitting(eta_measured)

print(k1, kd)
print(params.x[0], params.x[1])

Phi = np.zeros((2 * F.shape[0], 2))
# fabsim_py.phi_ode(Phi, V, P, F, stretch, 0.49, sigma, k0, params.x[0], params.x[1], 0.1, 10000)
fabsim_py.implicit_euler(Phi, V, P, F, fixed_dofs, stretch, 0.49, sigma, k0, params.x[0], params.x[1], 0.1, 0.1, k_post, 1000, k_tension)

phi_p = np.empty(F.shape[0])
eta = np.empty(F.shape[0])
dirs = np.empty((F.shape[0], 2))

for i in range(F.shape[0]):
  phi_p[i] = Phi[2 * i, 0] + Phi[2 * i + 1, 1]
  M = Phi[2 * i : 2 * (i + 1), :] / phi_p[i]
  w, v = np.linalg.eig(M)
  if w[0] > 0.5:
    dirs[i,:] = v[0]
    eta[i] = w[1]
  else:
    dirs[i,:] = v[1]
    eta[i] = w[0]

dirs[:,1] *= -1

k1 = params.x[0]
kd = params.x[1]

# re-estimate phi_p
I5 = fabsim_py.I5(V, P, F, orientation, eta_measured, stretch)
a = k1 / frac_f * sigma * (I5 - np.sqrt(I5))
b = k0 / frac_f + kd - 0.05 * a
c = -k0 * 0.05 / frac_f * np.ones_like(b)
phi_p_estimated = (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)

ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)

ps_mesh.add_scalar_quantity("phi_p_estimated", phi_p_estimated, defined_on='faces', enabled=True)
ps_mesh.add_scalar_quantity("eta_measured", eta_measured, defined_on='faces', enabled=True)
ps_mesh.add_scalar_quantity("eta", eta, defined_on='faces', enabled=True)
ps_mesh.add_scalar_quantity("phi_p", phi_p, defined_on='faces', enabled=True)
ps_mesh.add_vector_quantity("dirs", dirs, defined_on='faces', enabled=False)

ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
J = fabsim_py.J(V,P,F,stretch)
ps_mesh.add_scalar_quantity("J", J, defined_on='faces', enabled=False)
ps.show()
# print(np.average(np.abs(eta - eta_measured) / 0.5) * 100)
# print(np.average(np.abs(phi_p - phi_p_estimated) / 0.05) * 100)

In [ ]:
5e-2 / sigma

# Old

## Generate SDF data from masks

In [ ]:
from scipy.ndimage import distance_transform_edt

def generate_distance_from_mask(mask, show=False):
    mask = np.array(mask, dtype=bool)
    # Compute distance to the background (outside object)
    dist_outside = distance_transform_edt(mask, return_indices=False)

    # Compute distance to the foreground (inside object)
    dist_inside = distance_transform_edt(~mask, return_indices=False)

    # Signed Distance Field: negative inside, positive outside
    dist = (dist_outside + dist_inside) / world_coords_to_px

    if show:
        plt.imshow(dist, cmap='cividis')
        plt.colorbar(label='Signed Distance (mm)')
        plt.axis('off')
        plt.show()
    return dist

In [ ]:
n_cols = int(np.ceil(world_coords_to_px * 4.225)) + 10
n_rows = int(np.ceil(world_coords_to_px * 1.95)) + 10

# mean = np.empty((n_rows, n_cols))
dist = []

for i in range(1,4):
  mask = cv2.imread(f"data/mask{i}-boundary.png", cv2.IMREAD_GRAYSCALE) > 128
  height, width = mask.shape
  
  mask1 = np.full((n_rows, n_cols), False)
  mask1[n_rows - height//2:n_rows, 0:width//2] += np.flip(mask[0:height//2, 0:width//2], axis=1)
  dist.append(generate_distance_from_mask(mask1, show=False))

  mask2 = np.full((n_rows, n_cols), False)
  mask2[n_rows - height//2:n_rows, 0:width//2] += np.flip(mask[height//2:height, 0:width//2], axis=(0, 1))
  dist.append(generate_distance_from_mask(mask2, show=False))

  mask3 = np.full((n_rows, n_cols), False)
  mask3[n_rows - height//2:n_rows, 0:width//2] += mask[0:height//2, width//2:width]
  dist.append(generate_distance_from_mask(mask3, show=False))

  mask4 = np.full((n_rows, n_cols), False)
  mask4[n_rows - height//2:n_rows, 0:width//2] += np.flip(mask[height//2:height, width//2:width], axis=0)
  dist.append(generate_distance_from_mask(mask4, show=False))

mean = sum(dist) / len(dist)

plt.imshow(mean, cmap='cividis')
plt.colorbar(label='Signed Distance (mm)')
plt.axis('off')
plt.show()

R, G, B = to_8bit_rgb(mean)
img = np.array([R.T, G.T, B.T]).T
cv2.imwrite('data/distance-boundary.png', img)

In [ ]:
std = np.std(dist, axis=0)
plt.imshow(std, cmap='cividis')
plt.colorbar(label='Standard deviation (mm)')
plt.axis('off')
plt.show()
np.mean(std)

## Steady-state solution for fiber distribution

In [ ]:
stretch_factor = 1.1
Phi = np.zeros((F.shape[0], n))
P = P[:,:2]
V = P.copy()

for i in range(7):
  sigma = 100
  stretch = 1.25 + 0.25 * i
  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

  for i in range(3):
    Phi = fabsim_py.polymer_fraction_steady_state(sigma * Phi, 1, k1 / k0, kd / k0, frac_f, frac_s)
  ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)

  verts, faces = polygons_polar_plot(V, Phi, 0.25)
  ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
  ps.show()


## Convergence of solutions

Convergence of ODE

In [ ]:
face_id = 392
stretch_factor = 1.5
polymer_frac = np.zeros((F.shape[0], n))
V = V.copy()
phi_t = [np.zeros(n)]

for i in range(1000):
  fabsim_py.simulate_membrane(V, V, F, polymer_frac, fixed_dofs, stretch_factor, 0.25)
  sigmas = 1.0 * fabsim_py.fiber_stress(V, V / stretch_factor, F, n)
  fabsim_py.polymer_fraction_one_step(polymer_frac, sigmas, k0, k1, kd, frac_f, frac_s, dt)
  phi_t.append(polymer_frac[face_id, :].copy())

phi_t = np.array(phi_t)

for i in range(n):
  plt.plot(phi_t[:, i])
plt.show()

Convergence of steady-state solution

In [ ]:
stretch_factor = 1.5
polymer_frac = np.zeros((F.shape[0], n))
V = P.copy()
phi_t = [np.zeros(n)]

for i in range(10):
  fabsim_py.simulate_membrane(V, P, F, polymer_frac, fixed_dofs, stretch_factor, 0.25, 1.0, e0, e1)
  stress = fabsim_py.fiber_stress(V, P / stretch_factor, F, n, e0, e1)
  polymer_frac = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)
  phi_t.append(polymer_frac[face_id, :].copy())

phi_t = np.array(phi_t)

for i in range(n):
  plt.plot(phi_t[:, i])
plt.show()

Polar plot for "ground truth" solution

In [ ]:
stretch_factor = 1.5
n = 32
polymer_frac = np.zeros((F.shape[0], n))
V = P.copy()

for i in range(3):
  fabsim_py.simulate_membrane(V, P, F, polymer_frac, fixed_dofs, stretch_factor, 0.25)
  stress = fabsim_py.fiber_stress(V, P / stretch_factor, F, n, e0, e1)
  polymer_frac = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

_, ax = plt.subplots(subplot_kw={'projection': 'polar'})
ax.plot([np.pi / n * i for i in range(3 * n)], np.tile(polymer_frac[face_id, :], 3))
ax.set_rmax(0.12)

plt.show()

Polar plot for fake "measured data" (= ground thruth + noise)

In [ ]:
phi_measured = polymer_frac + 0.01 * np.random.default_rng().random(polymer_frac.shape)

_, ax = plt.subplots(subplot_kw={'projection': 'polar'})
ax.plot([np.pi / n * i for i in range(3 * n)], np.tile(phi_measured[face_id, :], 3))
ax.set_rmax(0.12)

plt.show()

In [ ]:
params = fitting(phi_measured)

print(k1, kd, e0, e1)

k1 = params.x[0]
kd = params.x[1]
e0 = params.x[2]
e1 = params.x[3]

print(k1, kd, e0, e1)

phi_recovered = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

_, ax = plt.subplots(subplot_kw={'projection': 'polar'})
ax.plot([np.pi / n * i for i in range(3 * n)], np.tile(phi_recovered[face_id, :], 3))
ax.set_rmax(0.12)

plt.show()

## Load orientation and intensity data

Load orientation CSV data

In [ ]:
input_csv = 'data/Orientation_angles_72hrs_compact_S1_16x_16x4_tilescan.csv' 
orientation_array = read_csv_to_array(input_csv)

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/orientation1-raw.png', img)
IPython.display.Image(filename='data/orientation1-raw.png')

In [ ]:
input_csv = 'data/orientation_angles_S2_72hrs_compact_16x_16x5_tilescan_denoise.csv'
orientation_array = read_csv_to_array(input_csv)
orientation_array = orientation_array

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/orientation2-raw.png', img)
IPython.display.Image(filename='data/orientation2-raw.png')

In [ ]:
input_csv = 'data/orientation_angles_S3_72hrs_compact_16x_16x5_tilescan_denoise.csv'
orientation_array = read_csv_to_array(input_csv)
height, width = orientation_array.shape

# transfer to 4 channel image (to have transparency)
img = np.empty([height, width, 4])
img[:,:,0] = orientation_array
img[:,:,1] = np.zeros(orientation_array.shape)
img[:,:,2] = np.zeros(orientation_array.shape)
img[:,:,3] = np.clip(orientation_array, a_min=0, a_max=1e-6) * 255 / 1e-6

# Rotate image by 1 degree
rotation_matrix = cv2.getRotationMatrix2D(center=(width // 2, height // 2), angle=1, scale=1)
rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

# Add rotation angle to data
rotated_img[:,:,0] = np.mod(rotated_img[:,:,0] + np.pi / 180., np.pi)

# Convert to 8 bit RGB (aka BGR) format
R, G, B = to_8bit_rgb(rotated_img[:,:,0] * 256 / np.pi)
A = rotated_img[:,:,3]
rotated_img = np.array([R, G, B, A]).T
rotated_img = cv2.flip(rotated_img, 0)
 
cv2.imwrite('data/orientation3-raw.png', rotated_img)
IPython.display.Image(filename='data/orientation3-raw.png')

In [ ]:
input_csv = 'data/7D/reprocessed_orientation_angles_S2_day7diff_16x_16x4tilescan.csv'
orientation_array = read_csv_to_array(input_csv)
height, width = orientation_array.shape

# transfer to 4 channel image (to have transparency)
img = np.empty([height, width, 4])
img[:,:,0] = orientation_array
img[:,:,1] = np.zeros(orientation_array.shape)
img[:,:,2] = np.zeros(orientation_array.shape)
img[:,:,3] = np.clip(orientation_array, a_min=0, a_max=1e-6) * 255 / 1e-6

# Rotate image by 1 degree
rotation_matrix = cv2.getRotationMatrix2D(center=(width // 2, height // 2), angle=-2.2, scale=1)
rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

# Add rotation angle to data
rotated_img[:,:,0] = np.mod(rotated_img[:,:,0] - 2.2 * np.pi / 180., np.pi)

# Convert to 8 bit RGB (aka BGR) format
R, G, B = to_8bit_rgb(rotated_img[:,:,0] * 256 / np.pi)
A = rotated_img[:,:,3]
rotated_img = np.array([R, G, B, A]).T
rotated_img = cv2.flip(rotated_img, 0)
 
cv2.imwrite('data/7D/orientation2-raw.png', rotated_img)
IPython.display.Image(filename='data/7D/orientation2-raw.png')

In [ ]:
input_csv = 'data/3.5hrs/orientation_angles_MAX_actin_c2c12_collagen_gel_3.5hrs_compact_S1_16x_18x8_tilescan - Denoise_ai_rotated.csv'
orientation_array = read_csv_to_array(input_csv)
orientation_array = orientation_array

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/3.5hrs/orientation1-raw.png', img)
IPython.display.Image(filename='data/3.5hrs/orientation1-raw.png')

In [ ]:
input_csv = 'data/3.5hrs/orientation_angles_MAX_actin_c2c12_collagen_gel_3.5hrs_compact_S2_16x_18x8_tilescan - Denoise_ai_rotated.csv'
orientation_array = read_csv_to_array(input_csv)
height, width = orientation_array.shape

# transfer to 4 channel image (to have transparency)
img = np.empty([height, width, 4])
img[:,:,0] = orientation_array
img[:,:,1] = np.zeros(orientation_array.shape)
img[:,:,2] = np.zeros(orientation_array.shape)
img[:,:,3] = np.clip(orientation_array, a_min=0, a_max=1e-6) * 255 / 1e-6

# Rotate image by 1 degree
rotation_matrix = cv2.getRotationMatrix2D(center=(width // 2, height // 2), angle=-1.5, scale=1)
rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

# Add rotation angle to data
rotated_img[:,:,0] = np.mod(rotated_img[:,:,0] -1.5 * np.pi / 180., np.pi)

# Convert to 8 bit RGB (aka BGR) format
R, G, B = to_8bit_rgb(rotated_img[:,:,0] * 256 / np.pi)
A = rotated_img[:,:,3]
rotated_img = np.array([R, G, B, A]).T
rotated_img = cv2.flip(rotated_img, 0)

cv2.imwrite('data/3.5hrs/orientation2-raw.png', rotated_img)
IPython.display.Image(filename='data/3.5hrs/orientation2-raw.png')

In [ ]:
input_csv = 'data/3.5hrs/orientation_angles_MAX_actin_c2c12_collagen_gel_3.5hrs_compact_S3_16x_18x8 - Denoise_ai_rotated_right.csv'
orientation_array = read_csv_to_array(input_csv)
height, width = orientation_array.shape

# transfer to 4 channel image (to have transparency)
img = np.empty([height, width, 4])
img[:,:,0] = orientation_array
img[:,:,1] = np.zeros(orientation_array.shape)
img[:,:,2] = np.zeros(orientation_array.shape)
img[:,:,3] = np.clip(orientation_array, a_min=0, a_max=1e-6) * 255 / 1e-6

# Rotate image by 1 degree
rotation_matrix = cv2.getRotationMatrix2D(center=(width // 2, height // 2), angle=-2.5, scale=1)
rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

# Add rotation angle to data
rotated_img[:,:,0] = np.mod(rotated_img[:,:,0] -2.5 * np.pi / 180., np.pi)

# Convert to 8 bit RGB (aka BGR) format
R, G, B = to_8bit_rgb(rotated_img[:,:,0] * 256 / np.pi)
A = rotated_img[:,:,3]
rotated_img = np.array([R, G, B, A]).T
rotated_img = cv2.flip(rotated_img, 0)
 
cv2.imwrite('data/3.5hrs/orientation3-raw.png', rotated_img)
IPython.display.Image(filename='data/3.5hrs/orientation3-raw.png')

In [ ]:
input_img = np.array(cv2.imread('data/3.5hrs/orientation3.png', cv2.IMREAD_UNCHANGED), dtype=np.float32)
mask = (input_img[:,:,3:4] > 0).astype(np.float64)

# Define a structuring element (kernel)
kernel = np.ones((5,5), np.uint8)

# Apply dilation
dilated_mask = cv2.dilate(mask, kernel, iterations=1)


cv2.imwrite('data/3.5hrs/mask3.png', dilated_mask * 255 / np.max(dilated_mask))
IPython.display.Image(filename='data/3.5hrs/mask3.png')

In [ ]:
input_csv = 'data/3 post/orientation_angles_actin-MAX_c2c12_collagen_gel_3post_S1_16x_ROI_7x7tilescan_zstack - Denoise_ai.csv'
orientation_array = read_csv_to_array(input_csv)
orientation_array = orientation_array

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/3 post/orientation1-raw.png', img)
IPython.display.Image(filename='data/3 post/orientation1-raw.png')

In [ ]:
input_csv = 'data/3 post/orientation_angles_actin-MAX_c2c12_collagen_gel_3post_S2_16x_ROI_8x8tilescan_zstack - Denoise_ai.csv'
orientation_array = read_csv_to_array(input_csv)
orientation_array = orientation_array

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/3 post/orientation2-raw.png', img)
IPython.display.Image(filename='data/3 post/orientation2-raw.png')

In [ ]:
input_csv = 'data/3 post/orientation_angles_actin-MAX_c2c12_collagen_gel_3post_S3_16x_ROI_7x7tilescan_zstack - Denoise_ai.csv'
orientation_array = read_csv_to_array(input_csv)
orientation_array = orientation_array

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/3 post/orientation3-raw.png', img)
IPython.display.Image(filename='data/3 post/orientation3-raw.png')

In [ ]:
input_csv = 'data/3 post/orientation_angles_actin-MAX_c2c12_collagen_gel_3post_S4_16x_ROI_8x8tilescan_zstack - Denoise_ai.csv'
orientation_array = read_csv_to_array(input_csv)
orientation_array = orientation_array

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/3 post/orientation4-raw.png', img)
IPython.display.Image(filename='data/3 post/orientation4-raw.png')

In [ ]:
img = cv2.imread(f'data/3 post/orientation3_dispersion.png', cv2.IMREAD_UNCHANGED)
mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
# array = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])
# print(np.average(array * 360. / 256 - 180, weights=mask))
histr = cv2.calcHist([img],[0],mask,[256],[0,256])
plt.plot(histr)
plt.xlim([0,256])

# for i in range(1,4):
#     img = cv2.imread(f'data/orientation{i}.png', cv2.IMREAD_UNCHANGED)
#     mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
#     histr = cv2.calcHist([img],[0],mask,[256],[0,256])
#     plt.plot(histr)
#     plt.xlim([0,256])

plt.show()

In [ ]:
input_img = cv2.imread(data_path + 'orientation1.png')
height, width, _ = input_img.shape

orientation = np.zeros((height, width))
complex_a = np.zeros((height, width))
complex_b = np.zeros((height, width))

alpha = np.zeros((height, width))

for i in range(1,4):
  input_img = np.array(cv2.imread(data_path + 'orientation{i}.png', cv2.IMREAD_UNCHANGED), dtype=np.float32)
  input_img = input_img * (input_img[:,:,3:4] > 0)

  alpha += (input_img[:,:,3] > 0).astype(float)
  complex_a += np.cos(2 * from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * np.pi / 256) * (input_img[:,:,3] > 0)
  complex_b += np.sin(2 * from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * np.pi / 256) * (input_img[:,:,3] > 0)

complex_a /= np.clip(alpha, a_min=1, a_max=1000)
complex_b /= np.clip(alpha, a_min=1, a_max=1000)

complex_ab = np.sqrt(complex_a + 1j * complex_b)
orientation = np.mod(np.angle(complex_ab), np.pi) * 256 / np.pi

R, G, B = to_8bit_rgb(orientation)
A = (alpha > 0) * 255
img = np.array([R.T, G.T, B.T, A.T]).T

cv2.imwrite(data_path + 'orientation-combined.png', img)
IPython.display.Image(filename=data_path + 'orientation-combined.png')

In [ ]:
n_cols = int(np.ceil(1287.2 * 4.225)) + 10
n_rows = int(np.ceil(1287.2 * 1.95)) + 10

alpha = np.zeros((n_rows, n_cols))
complex_a = np.zeros((n_rows, n_cols))
complex_b = np.zeros((n_rows, n_cols))

input_img = np.array(cv2.imread(data_path + 'orientation-combined.png', cv2.IMREAD_UNCHANGED), dtype=np.float32)

height, width, _ = input_img.shape

data = np.cos(2 * from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * np.pi / 256) * (input_img[:,:,3] > 0)
add_mirror_images(complex_a, data, data, height, width)

data = np.sin(2 * from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * np.pi / 256) * (input_img[:,:,3] > 0)
add_mirror_images(complex_b, data, -data, height, width)

data = (input_img[:,:,3] > 0).astype(float)
add_mirror_images(alpha, data, data, height, width)

complex_a /= np.clip(alpha, a_min=1, a_max=1000000)
complex_b /= np.clip(alpha, a_min=1, a_max=1000000)
complex_ab = np.sqrt(complex_a + 1j * complex_b)

orientation = np.mod(np.angle(complex_ab), np.pi) * 256 / np.pi
R, G, B = to_8bit_rgb(orientation)
A = (alpha > 0) * 255
orientation = np.array([R.T, G.T, B.T, A.T]).T

n_rows, n_cols, _ = orientation.shape
full_image = np.empty((2 * n_rows, 2 * n_cols, 4))

full_image[0:n_rows, 0:n_cols, :] = np.flip(orientation, axis=1)
full_image[n_rows:2*n_rows, 0:n_cols, :] = np.flip(orientation, axis=(0, 1))
full_image[0:n_rows, n_cols:2*n_cols, :] = orientation
full_image[n_rows:2*n_rows, n_cols:2*n_cols, :] = np.flip(orientation, axis=0)

full_image[0:n_rows, 0:n_cols, :3] = (255 - full_image[0:n_rows, 0:n_cols, :3])
full_image[n_rows:2*n_rows, n_cols:2*n_cols, :3] = (255 - full_image[n_rows:2*n_rows, n_cols:2*n_cols, :3])

cv2.imwrite(data_path + 'orientation.png', full_image)
IPython.display.Image(filename=data_path + 'orientation.png')

Load pixel intensity CSV data

In [ ]:
input_csv = 'data/pixel_intensity_72hrs_compact_S1_16x_16x4_tilescan.csv'  # Replace with your filename
intensity_array = read_csv_to_array(input_csv)

R, G, B = to_8bit_rgb(intensity_array.T * 256 / np.max(intensity_array))
A = (R > 4) * 255
img = np.array([R, G, B, A]).T
# img = cv2.flip(intensity_array, 0)

# i, j = np.unravel_index(intensity_array.argmax(), intensity_array.shape)
# print(np.max(intensity_array))
# print(i, j)
# print(intensity_array[i-5:i+5,j-5:j+5])
cv2.imwrite('data/intensity1-raw.png', img)
IPython.display.Image(filename='data/intensity1-raw.png')

In [ ]:
input_csv = 'data/pixel_intensity_S2_72hrs_compact_16x_16x5_tilescan_denoise.csv'  # Replace with your filename
intensity_array = read_csv_to_array(input_csv)

R, G, B = to_8bit_rgb(intensity_array * 256 / np.max(intensity_array))
A = (R > 4) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
print(np.argmax(intensity_array))
cv2.imwrite('data/intensity2-raw.png', img)
IPython.display.Image(filename='data/intensity2-raw.png')

In [ ]:
input_csv = 'data/pixel_intensity_S3_72hrs_compact_16x_16x5_tilescan_denoise.csv'  # Replace with your filename
intensity_array = read_csv_to_array(input_csv)
height, width = intensity_array.shape

# transfer to 4 channel image (to have transparency)
img = np.empty([height, width, 4])
img[:,:,0] = 256 * intensity_array / np.max(intensity_array)
img[:,:,1] = np.zeros(intensity_array.shape)
img[:,:,2] = np.zeros(intensity_array.shape)
img[:,:,3] = (img[:,:,0] > 5) * 256

# Rotate image by 1 degree
rotation_matrix = cv2.getRotationMatrix2D(center=(width // 2, height // 2), angle=1, scale=1)
rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

# Convert to 8 bit RGB (aka BGR) format
R, G, B = to_8bit_rgb(rotated_img[:,:,0])
A = rotated_img[:,:,3]
rotated_img = np.array([R, G, B, A]).T
rotated_img = cv2.flip(rotated_img, 0)
 
cv2.imwrite('data/intensity3-raw.png', rotated_img)
print(np.argmax(intensity_array))
IPython.display.Image(filename='data/intensity3-raw.png')

In [ ]:
img = cv2.imread(f'data/intensity.png', cv2.IMREAD_UNCHANGED)
mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
# array = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])
# print(np.average(array * 360. / 256 - 180, weights=mask))
histr = cv2.calcHist([img],[0],mask,[256],[0,256])
plt.plot(histr)
plt.xlim([0,256])

for i in range(1,4):
    img = cv2.imread(f'data/intensity{i}.png', cv2.IMREAD_UNCHANGED)
    mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
    histr = cv2.calcHist([img],[0],mask,[256],[0,256])
    plt.plot(histr)
    plt.xlim([0,256])

plt.show()

In [ ]:
n_cols = int(np.ceil(1287.2 * 4.225)) + 10
n_rows = int(np.ceil(1287.2 * 1.95)) + 10

intensity = np.full((n_rows, n_cols), 0.0)
alpha = np.full((n_rows, n_cols), 0.0)
for i in range(1,4):
  input_img = np.array(cv2.imread(f"data/intensity{i}.png", cv2.IMREAD_UNCHANGED), dtype=np.float32)
  img = input_img[:,:,:3] * input_img[:,:,3:4]
  data = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])

  height, width, _ = img.shape
  
  intensity[n_rows - height//2:n_rows, 0:width//2] += np.flip(data[0:height//2, 0:width//2], axis=1)
  intensity[n_rows - height//2:n_rows, 0:width//2] += np.flip(data[height//2:height, 0:width//2], axis=(0, 1))
  intensity[n_rows - height//2:n_rows, 0:width//2] += data[0:height//2, width//2:width]
  intensity[n_rows - height//2:n_rows, 0:width//2] += np.flip(data[height//2:height, width//2:width], axis=0)

  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[0:height//2, 0:width//2, 3], axis=1)
  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[height//2:height, 0:width//2, 3], axis=(0, 1))
  alpha[n_rows - height//2:n_rows, 0:width//2] += input_img[0:height//2, width//2:width, 3]
  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[height//2:height, width//2:width, 3], axis=0)

intensity /= np.clip(alpha, a_min=1, a_max=1000000)
R, G, B = to_8bit_rgb(intensity)
A = (alpha > 0) * 255
intensity = np.array([R.T, G.T, B.T, A.T]).T

n_rows, n_cols, _ = intensity.shape
full_image = np.empty((2 * n_rows, 2 * n_cols, 4))

full_image[0:n_rows, 0:n_cols, :] = np.flip(intensity, axis=1)
full_image[n_rows:2*n_rows, 0:n_cols, :] = np.flip(intensity, axis=(0, 1))
full_image[0:n_rows, n_cols:2*n_cols, :] = intensity
full_image[n_rows:2*n_rows, n_cols:2*n_cols, :] = np.flip(intensity, axis=0)

cv2.imwrite('data/intensity.png', full_image)
IPython.display.Image(filename='data/intensity.png')

## Get mean and dispersion for orientation data

In [ ]:
img = cv2.imread(data_path + 'orientation2.png', cv2.IMREAD_UNCHANGED)

data = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * np.pi / 256
cos_data = np.cos(2 * data) * img[:,:,3]
sin_data = np.sin(2 * data) * img[:,:,3]

blur = 50

# normalized convolution of image with mask
sin_smooth = gaussian_filter(sin_data, sigma = blur)
cos_smooth = gaussian_filter(cos_data, sigma = blur)

weights = np.clip(gaussian_filter(img[:,:,3].astype(np.float64), sigma = blur), a_min=1, a_max=1000) 
sin_smooth /= weights
cos_smooth /= weights

complex_ab = np.sqrt(cos_smooth + 1j * sin_smooth)

mean_orientation = np.mod(np.angle(complex_ab), np.pi) * 256 / 255 * img[:,:,3] / np.pi

R, G, B = to_8bit_rgb(mean_orientation)
A = img[:,:,3]
new_img = np.array([R.T, G.T, B.T, A.T]).T

cv2.imwrite(data_path + 'orientation2_mean.png', new_img)
IPython.display.Image(filename=data_path + 'orientation2_mean.png')

In [ ]:
import numpy as np
from numba import njit, prange
import scipy.special

@njit(parallel=True, fastmath=True)
def integrate_field(P, x, dx):
    ny, nx = P.shape
    n_x = len(x)
    F = np.empty_like(P)

    for i in prange(ny):               # parallel loop over rows
        for j in range(nx):            # inner loop over columns
            p = P[i, j]
            acc = 0.0
            for k in range(n_x - 1):   # trapezoidal rule
                f1 = np.exp(p * np.cos(2 * x[k])) * np.sin(x[k])**2
                f2 = np.exp(p * np.cos(2 * x[k+1])) * np.sin(x[k+1])**2
                acc += 0.5 * (f1 + f2)
            F[i, j] = acc * dx
    return F

In [ ]:
# mean resultant length
R = np.sqrt(sin_smooth**2 + cos_smooth**2)

kappa = np.zeros_like(R)

# approximation from Best & Fisher [1981] http://dx.doi.org/10.1080/03610918108812225
lower_R = R[R < 0.53]
mid_R = R[(R > 0.53) & (R < 0.85)]
upper_R = R[R > 0.85]

kappa[R < 0.53] = 2 * lower_R + lower_R**3 + (5 * lower_R**5) / 6
kappa[(R > 0.53) & (R < 0.85)] = -0.4 + 1.39 * mid_R + 0.43 / (1 - mid_R)
kappa[R > 0.85] = 1 / (upper_R**3 - 4 * upper_R**2 + 3 * upper_R)


theta = np.linspace(-np.pi/2, np.pi/2, 256)

eta = integrate_field(kappa, theta, theta[1] - theta[0])
eta /= scipy.special.iv(0, kappa) * np.pi

r, g, b = to_8bit_rgb(eta * 256 / 0.5)
A = img[:,:,3]

new_img = np.array([r.T, g.T, b.T, A.T]).T

cv2.imwrite(data_path + 'orientation2_dispersion.png', new_img)
IPython.display.Image(filename=data_path + 'orientation2_dispersion.png')

In [ ]:
for i in range(1,4):
  img = cv2.imread(f'{data_path}orientation{i}.png', cv2.IMREAD_UNCHANGED)

  data = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * np.pi / 256
  cos_data = np.cos(2 * data) * img[:,:,3]
  sin_data = np.sin(2 * data) * img[:,:,3]

  blur = 50

  # normalized convolution of image with mask
  sin_smooth = gaussian_filter(sin_data, sigma = blur)
  cos_smooth = gaussian_filter(cos_data, sigma = blur)

  weights = np.clip(gaussian_filter(img[:,:,3].astype(np.float64), sigma = blur), a_min=1, a_max=1000) 
  sin_smooth /= weights
  cos_smooth /= weights

  # mean resultant length
  R = np.sqrt(sin_smooth**2 + cos_smooth**2)

  kappa = np.zeros_like(R)

  # approximation from Best & Fisher [1981] http://dx.doi.org/10.1080/03610918108812225
  lower_R = R[R < 0.53]
  mid_R = R[(R > 0.53) & (R < 0.85)]
  upper_R = R[R > 0.85]

  kappa[R < 0.53] = 2 * lower_R + lower_R**3 + (5 * lower_R**5) / 6
  kappa[(R > 0.53) & (R < 0.85)] = -0.4 + 1.39 * mid_R + 0.43 / (1 - mid_R)
  kappa[R > 0.85] = 1 / (upper_R**3 - 4 * upper_R**2 + 3 * upper_R)

  theta = np.linspace(-np.pi/2, np.pi/2, 256)

  eta = integrate_field(kappa, theta, theta[1] - theta[0])
  eta /= scipy.special.iv(0, kappa) * np.pi

  r, g, b = to_8bit_rgb(eta * 256 / 0.5)
  A = img[:,:,3]

  new_img = np.array([r.T, g.T, b.T, A.T]).T

  cv2.imwrite(f'{data_path}orientation_dispersion{i}.png', new_img)
IPython.display.Image(filename='data/orientation_dispersion1.png')

In [ ]:
input_img = cv2.imread(data_path + 'orientation_dispersion1.png')
height, width, _ = input_img.shape

n_cols = int(np.ceil(1287.2 * 4.225)) + 10
n_rows = int(np.ceil(1287.2 * 1.95)) + 10

eta = np.zeros((n_rows, n_cols))
alpha = np.zeros((n_rows, n_cols))
eta_combined = np.zeros((height, width))
alpha_combined = np.zeros((height, width))

for i in range(1,4):
  input_img = np.array(cv2.imread(f'{data_path}orientation_dispersion{i}.png', cv2.IMREAD_UNCHANGED), dtype=np.float32)
  input_img = input_img * (input_img[:,:,3:4] > 0)

  alpha_combined += (input_img[:,:,3] > 0).astype(float)
  eta_combined += from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * 0.5 / 256

# eta_combined /= np.clip(alpha_combined, a_min=1, a_max=1000)

add_mirror_images(eta, eta_combined, eta_combined, height, width)
add_mirror_images(alpha, alpha_combined, alpha_combined, height, width)

eta /= np.clip(alpha, a_min=1, a_max=1000)
print(np.min(eta), np.max(eta))

R, G, B = to_8bit_rgb(eta * 256 / 0.5)
A = (alpha > 0) * 255
img = np.array([R.T, G.T, B.T, A.T]).T

full_image = np.empty((2 * n_rows, 2 * n_cols, 4))

full_image[0:n_rows, 0:n_cols, :] = np.flip(img, axis=1)
full_image[n_rows:2*n_rows, 0:n_cols, :] = np.flip(img, axis=(0, 1))
full_image[0:n_rows, n_cols:2*n_cols, :] = img
full_image[n_rows:2*n_rows, n_cols:2*n_cols, :] = np.flip(img, axis=0)

cv2.imwrite(data_path + 'orientation-dispersion-combined.png', full_image)
IPython.display.Image(filename=data_path + 'orientation-dispersion-combined.png')

In [ ]:
img = cv2.imread('data/orientation-dispersion-combined.png', cv2.IMREAD_UNCHANGED)
mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
histr = cv2.calcHist([img],[0],mask,[256],[0,256])
plt.plot(histr)
plt.xlim([0,256])

img = cv2.imread('data/orientation_dispersion.png', cv2.IMREAD_UNCHANGED)
mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
histr = cv2.calcHist([img],[0],mask,[256],[0,256])
plt.plot(histr)
plt.xlim([0,256])

plt.show()

In [ ]:
plt.plot(kappa.ravel(), eta.ravel())
plt.xlabel("$\kappa$")
plt.ylabel("$\eta$")
plt.show()

In [ ]:
# Show image with HSV colormap
img = cv2.imread(data_path + 'orientation2_mean.png', cv2.IMREAD_UNCHANGED)
mean_orientation = np.mod(from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * np.pi / 255 + np.pi/2, np.pi) * (img[:,:,3] > 0)
np.max(mean_orientation)

plt.imshow(mean_orientation, cmap='twilight', origin='lower', aspect='equal')
plt.colorbar(label="Angle (radians)")
plt.title("Mean F-actin orientation (S2)")
plt.tight_layout()
plt.show()

In [ ]:
img = cv2.imread(data_path + 'orientation2_dispersion.png', cv2.IMREAD_UNCHANGED)
eta = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * 0.5 / 256

plt.imshow(eta, cmap='turbo', origin='lower', vmin=0, vmax=0.5, extent=[0, mean_orientation.shape[1] / world_coords_to_px, 0, mean_orientation.shape[0] / world_coords_to_px], aspect='equal', alpha=img[:,:,3] / 255)
plt.colorbar(label="$\eta$")
plt.title("Dispersion measure $\eta$ (S2)")
plt.tight_layout()
plt.show()

## Simulation comparisons

Shape from simulated fibers

In [ ]:
# Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.3, n)
Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.6, n)

# normalize Phi
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16

verts, faces = vertex_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 

Shape from measured fibers

In [ ]:
# get orientation
Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.6, n)
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16

# sim
stretch_factor = 2.5
Phi_sim = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3
print(np.sum(Phi_sim))

fabsim_py.simulate_membrane(V, P, F, Phi_sim, fixed_dofs, stretch_factor, 0.49)

# display
verts, faces = vertex_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 
# ps.show()

Shape from measured fibers with intensity scaling

In [ ]:
# get intensity and orientation
intensity = fabsim_py.image_data_to_mesh(V, F, intensity_array, world_coords_to_px)
Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.6, n)

# rescale intensity
intensity = intensity / np.max(intensity)

# rescale Phi
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16
Phi = Phi * intensity

# sim
stretch_factor = 2.5
Phi_sim = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3

fabsim_py.simulate_membrane(V, P, F, Phi_sim, fixed_dofs, stretch_factor, 0.49)

# display
verts, faces = vertex_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 
# ps.show()

## Fitting

In [ ]:
# get orientation
Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.6, n)
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16

# sim
stretch_factor = 2.5
phi_measured = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3

fabsim_py.simulate_membrane(V, P, F, phi_measured, fixed_dofs, stretch_factor, 0.49)

params = fitting(phi_measured)

print(k1, kd, e0, e1)

k1 = params.x[0]
kd = params.x[1]
e0 = params.x[2]
e1 = params.x[3]

print(k1, kd, e0, e1)

phi_recovered = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

fabsim_py.simulate_membrane(V, P, F, phi_recovered, fixed_dofs, stretch_factor, 0.49)

# display
verts, faces = polygons_polar_plot(V, phi_measured, 0.25)
ps.register_surface_mesh("phi_measured", verts, faces, enabled=False, color=(213/255, 202/255, 42/255))
verts, faces = polygons_polar_plot(V, phi_recovered, 0.25)
ps.register_surface_mesh("phi_recovered", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 
# ps.show()

## Tissue mesh extraction

Generate intensity mask

In [ ]:
mask = np.array(intensity_array > 300, dtype=float)

# Define a structuring element (kernel)
kernel = np.ones((25,25), np.uint8)

# Apply dilation
dilated_mask = cv2.dilate(mask, kernel, iterations=1)


cv2.imwrite('data/temp.png', dilated_mask * 255 / np.max(dilated_mask))
IPython.display.Image(filename='data/temp.png')

In [ ]:
mask = np.array(intensity_array2 > 300, dtype=float)

# Define a structuring element (kernel)
kernel = np.ones((25,25), np.uint8)

# Apply dilation
dilated_mask2 = cv2.dilate(mask, kernel, iterations=1)


cv2.imwrite('data/temp.png', dilated_mask2 * 255 / np.max(dilated_mask2))
IPython.display.Image(filename='data/temp.png')

In [ ]:
mask = np.array(intensity_array3 > 150, dtype=float)

# Define a structuring element (kernel)
kernel = np.ones((25,25), np.uint8)

# Apply dilation
dilated_mask3 = cv2.dilate(mask, kernel, iterations=1)

cv2.imwrite('data/temp.png', dilated_mask3 * 255 / np.max(dilated_mask3))
IPython.display.Image(filename='data/temp.png')